In [25]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Basic DataFrame Operations
1. Load the sales.csv and customer.csv files into separate DataFrames.
2. Display the schema of both DataFrames.
3. Show the first 5 rows from the sales DataFrame.
4. Count the number of rows and columns in the customer DataFrame.

In [3]:
spark = SparkSession.builder.appName("Assignment").getOrCreate()

In [7]:
sales_df = spark.read.csv("/content/sales.txt",header=True,inferSchema=True)
customer_df = spark.read.csv("/content/customer.txt",header=True,inferSchema=True)


In [ ]:
customer_df.printSchema()
sales_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)

root
 |-- sales_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- region: string (nullable = true)



In [ ]:
sales_df.show(5)

+--------+-----------+-------+------+----------+------+
|sales_id|customer_id|product|amount| sale_date|region|
+--------+-----------+-------+------+----------+------+
|       1|        101| Laptop| 50000|2023-01-15| North|
|       2|        102| Mobile| 15000|2023-02-10| South|
|       3|        103| Tablet| 20000|2023-03-05|  West|
|       4|        104| Laptop| 55000|2023-03-15|  East|
|       5|        105|Desktop| 40000|2023-04-20| North|
+--------+-----------+-------+------+----------+------+
only showing top 5 rows


In [ ]:
rows = customer_df.count()
columns = len(customer_df.columns)

## Data Cleaning
5. Remove duplicate rows from the sales DataFrame based on
customer_id,product,amount,sale_date,region columns
6. Drop rows where any column in the customer DataFrame has null values.
7. Replace null values in the amount column of the sales DataFrame with 0.
8. Replace null values in the email column of the customer DataFrame with the value
"unknown".

In [ ]:
sales_df = sales_df.dropDuplicates(
    ["customer_id","product","amount","sale_date","region"])

In [ ]:
customer_df = customer_df.dropna()

In [ ]:
sales_df = sales_df.fillna({"amount":0})

In [ ]:
customer_df = customer_df.fillna({"email":"unknown"})

## Column Manipulation
9. Add a new column discounted_amount to the sales DataFrame that applies a 10%
discount on amount.
10. Rename the city column in the customer DataFrame to customer_city.
11. Drop the region column from the sales DataFrame.
12. Create a new column customer_age_category in the customer DataFrame based on age:
a. "Youth" for age < 30
b. "Adult" for 30 <= age < 50
c. "Senior" for age >= 50

In [ ]:
a = sales_df.withColumn("discounted_amount",col("amount")*0.9)

a.show()

+--------+-----------+-------+------+----------+------+-----------------+
|sales_id|customer_id|product|amount| sale_date|region|discounted_amount|
+--------+-----------+-------+------+----------+------+-----------------+
|       1|        101| Laptop| 50000|2023-01-15| North|          45000.0|
|       3|        103| Tablet| 20000|2023-03-05|  West|          18000.0|
|       7|        102| Laptop| 60000|2023-06-15|  East|          54000.0|
|      10|        105| Laptop| 70000|2023-09-25| North|          63000.0|
|       5|        105|Desktop| 40000|2023-04-20| North|          36000.0|
|       2|        102| Mobile| 15000|2023-02-10| South|          13500.0|
|       9|        104|Desktop| 45000|2023-08-10|  West|          40500.0|
|       6|        101| Mobile| 15000|2023-05-10| South|          13500.0|
|       8|        103| Tablet| 20000|2023-07-05| North|          18000.0|
|       4|        104| Laptop| 55000|2023-03-15|  East|          49500.0|
+--------+-----------+-------+------+-

In [ ]:
customer_df.withColumnRenamed("city","customer_city")

DataFrame[customer_id: int, customer_name: string, email: string, age: int, customer_city: string]

In [ ]:
temp = sales_df.drop("region")

In [ ]:
b = customer_df.withColumn("customer_age_category",
                           when(col("age")<30,"Youth") \
                           .when((col("age")>=30)& (col("age")<50),"Adult")\
                           .otherwise("Senior"))
b.show()

+-----------+-------------+--------------------+---+---------+---------------------+
|customer_id|customer_name|               email|age|     city|customer_age_category|
+-----------+-------------+--------------------+---+---------+---------------------+
|        101|  Arun Sharma|arun.sharma@email...| 28|    Delhi|                Youth|
|        102|  Meena Verma|meena.verma@email...| 34|   Mumbai|                Adult|
|        103|  Rahul Yadav|rahul.yadav@email...| 30|Bangalore|                Adult|
|        104|  Priya Patel|priya.patel@email...| 27|Ahmedabad|                Youth|
|        105|  Sneha Reddy|sneha.reddy@email...| 29|Hyderabad|                Youth|
|        106|   Vikas Jain|vikas.jain@email.com| 31|  Chennai|                Adult|
|        107|     Amit Roy|  amit.roy@email.com| 35|  Kolkata|                Adult|
+-----------+-------------+--------------------+---+---------+---------------------+



## Filtering
13. Filter the sales DataFrame to show only rows where amount is greater than 50,000.
14. Filter the customer DataFrame to show customers aged between 25 and 30.
15. Identify all customers who have made purchases in more than one region.
16. Filter the top 3 sales based on amount for each product.

In [ ]:
sales_df.filter(col("amount")> 50000).show()

+--------+-----------+-------+------+----------+------+
|sales_id|customer_id|product|amount| sale_date|region|
+--------+-----------+-------+------+----------+------+
|       7|        102| Laptop| 60000|2023-06-15|  East|
|      10|        105| Laptop| 70000|2023-09-25| North|
|       4|        104| Laptop| 55000|2023-03-15|  East|
+--------+-----------+-------+------+----------+------+



In [ ]:
customer_df.filter((col("age")>25) & (col("age")<30)).show()

+-----------+-------------+--------------------+---+---------+
|customer_id|customer_name|               email|age|     city|
+-----------+-------------+--------------------+---+---------+
|        101|  Arun Sharma|arun.sharma@email...| 28|    Delhi|
|        104|  Priya Patel|priya.patel@email...| 27|Ahmedabad|
|        105|  Sneha Reddy|sneha.reddy@email...| 29|Hyderabad|
+-----------+-------------+--------------------+---+---------+



In [ ]:
m_regions = (
    sales_df.groupBy("customer_id")
            .agg(count("region").alias("region_count"))
            .filter(col("region_count") > 1)
            .select("customer_id")
)

m_regions.show()

+-----------+
|customer_id|
+-----------+
|        101|
|        103|
|        102|
|        105|
|        104|
+-----------+



In [ ]:
spec = Window.partitionBy("sales_id").orderBy(desc("amount"))

top3 = (
    sales_df.withColumn("rank", row_number().over(spec))
            .filter(col("rank") <= 3)
            .drop("rank")
)

top3.show()

+--------+-----------+-------+------+----------+------+
|sales_id|customer_id|product|amount| sale_date|region|
+--------+-----------+-------+------+----------+------+
|       1|        101| Laptop| 50000|2023-01-15| North|
|       2|        102| Mobile| 15000|2023-02-10| South|
|       3|        103| Tablet| 20000|2023-03-05|  West|
|       4|        104| Laptop| 55000|2023-03-15|  East|
|       5|        105|Desktop| 40000|2023-04-20| North|
|       6|        101| Mobile| 15000|2023-05-10| South|
|       7|        102| Laptop| 60000|2023-06-15|  East|
|       8|        103| Tablet| 20000|2023-07-05| North|
|       9|        104|Desktop| 45000|2023-08-10|  West|
|      10|        105| Laptop| 70000|2023-09-25| North|
+--------+-----------+-------+------+----------+------+



## Joins
17. Perform an inner join between sales and customer DataFrames on customer_id.
18. Perform a left join to include all records from sales and matching records from
customer.
19. Perform a full outer join between sales and customer DataFrames.
20. Identify customers who have not made any purchases by performing an anti-join.

In [ ]:
j1 = sales_df.join(customer_df,on ="customer_id",how="inner")

j1.show()

+-----------+--------+-------+------+----------+------+-------------+--------------------+---+---------+
|customer_id|sales_id|product|amount| sale_date|region|customer_name|               email|age|     city|
+-----------+--------+-------+------+----------+------+-------------+--------------------+---+---------+
|        101|       1| Laptop| 50000|2023-01-15| North|  Arun Sharma|arun.sharma@email...| 28|    Delhi|
|        103|       3| Tablet| 20000|2023-03-05|  West|  Rahul Yadav|rahul.yadav@email...| 30|Bangalore|
|        102|       7| Laptop| 60000|2023-06-15|  East|  Meena Verma|meena.verma@email...| 34|   Mumbai|
|        105|      10| Laptop| 70000|2023-09-25| North|  Sneha Reddy|sneha.reddy@email...| 29|Hyderabad|
|        105|       5|Desktop| 40000|2023-04-20| North|  Sneha Reddy|sneha.reddy@email...| 29|Hyderabad|
|        102|       2| Mobile| 15000|2023-02-10| South|  Meena Verma|meena.verma@email...| 34|   Mumbai|
|        104|       9|Desktop| 45000|2023-08-10|  West|

In [ ]:
j2 = sales_df.join(customer_df,on ="customer_id",how="left")

j2.show()

+-----------+--------+-------+------+----------+------+-------------+--------------------+---+---------+
|customer_id|sales_id|product|amount| sale_date|region|customer_name|               email|age|     city|
+-----------+--------+-------+------+----------+------+-------------+--------------------+---+---------+
|        101|       1| Laptop| 50000|2023-01-15| North|  Arun Sharma|arun.sharma@email...| 28|    Delhi|
|        103|       3| Tablet| 20000|2023-03-05|  West|  Rahul Yadav|rahul.yadav@email...| 30|Bangalore|
|        102|       7| Laptop| 60000|2023-06-15|  East|  Meena Verma|meena.verma@email...| 34|   Mumbai|
|        105|      10| Laptop| 70000|2023-09-25| North|  Sneha Reddy|sneha.reddy@email...| 29|Hyderabad|
|        105|       5|Desktop| 40000|2023-04-20| North|  Sneha Reddy|sneha.reddy@email...| 29|Hyderabad|
|        102|       2| Mobile| 15000|2023-02-10| South|  Meena Verma|meena.verma@email...| 34|   Mumbai|
|        104|       9|Desktop| 45000|2023-08-10|  West|

In [ ]:
j3 = customer_df.join(sales_df,how="fullouter")

j3.show()

+-----------+-------------+--------------------+---+---------+--------+-----------+-------+------+----------+------+
|customer_id|customer_name|               email|age|     city|sales_id|customer_id|product|amount| sale_date|region|
+-----------+-------------+--------------------+---+---------+--------+-----------+-------+------+----------+------+
|        101|  Arun Sharma|arun.sharma@email...| 28|    Delhi|       1|        101| Laptop| 50000|2023-01-15| North|
|        102|  Meena Verma|meena.verma@email...| 34|   Mumbai|       1|        101| Laptop| 50000|2023-01-15| North|
|        103|  Rahul Yadav|rahul.yadav@email...| 30|Bangalore|       1|        101| Laptop| 50000|2023-01-15| North|
|        104|  Priya Patel|priya.patel@email...| 27|Ahmedabad|       1|        101| Laptop| 50000|2023-01-15| North|
|        105|  Sneha Reddy|sneha.reddy@email...| 29|Hyderabad|       1|        101| Laptop| 50000|2023-01-15| North|
|        106|   Vikas Jain|vikas.jain@email.com| 31|  Chennai|  

In [ ]:
j3 = customer_df.join(sales_df,on="customer_id",how="anti").show()

+-----------+-------------+--------------------+---+-------+
|customer_id|customer_name|               email|age|   city|
+-----------+-------------+--------------------+---+-------+
|        106|   Vikas Jain|vikas.jain@email.com| 31|Chennai|
|        107|     Amit Roy|  amit.roy@email.com| 35|Kolkata|
+-----------+-------------+--------------------+---+-------+



## Aggregations
21. Calculate the total sales amount for each product.
22. Find the average age of customers in the customer DataFrame.
23. Calculate the maximum and minimum sales amounts in the sales DataFrame.
24. Group the customer DataFrame by customer_city and count the number of customers in
each city.

In [9]:
sum = sales_df.groupBy("product").agg(sum(col("amount"))).show()

+-------+-----------+
|product|sum(amount)|
+-------+-----------+
| Laptop|     235000|
| Mobile|      30000|
| Tablet|      40000|
|Desktop|      85000|
+-------+-----------+



In [11]:
avg_age = customer_df.agg(avg("age")).alias("avg_age").show()

+------------------+
|          avg(age)|
+------------------+
|30.571428571428573|
+------------------+



In [19]:
sales_df.agg(
    min(col("amount")).alias("min_amount"),
    max(col("amount")).alias("max_amount")
).show()

+----------+----------+
|min_amount|max_amount|
+----------+----------+
|     15000|     70000|
+----------+----------+



In [20]:
d = customer_df.groupBy("city").agg(count("customer_id").alias("count")).show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    1|
|  Chennai|    1|
|   Mumbai|    1|
|Ahmedabad|    1|
|  Kolkata|    1|
|    Delhi|    1|
|Hyderabad|    1|
+---------+-----+



## Sorting
25. Sort the sales DataFrame by amount in descending order.
26. Sort the customer DataFrame by age in ascending order.

In [27]:
s1 = sales_df.orderBy((col("amount")).desc()).show()

+--------+-----------+-------+------+----------+------+
|sales_id|customer_id|product|amount| sale_date|region|
+--------+-----------+-------+------+----------+------+
|      10|        105| Laptop| 70000|2023-09-25| North|
|       7|        102| Laptop| 60000|2023-06-15|  East|
|       4|        104| Laptop| 55000|2023-03-15|  East|
|       1|        101| Laptop| 50000|2023-01-15| North|
|       9|        104|Desktop| 45000|2023-08-10|  West|
|       5|        105|Desktop| 40000|2023-04-20| North|
|       3|        103| Tablet| 20000|2023-03-05|  West|
|       8|        103| Tablet| 20000|2023-07-05| North|
|       2|        102| Mobile| 15000|2023-02-10| South|
|       6|        101| Mobile| 15000|2023-05-10| South|
+--------+-----------+-------+------+----------+------+



In [28]:
s2 = customer_df.orderBy(col("age").asc()).show()

+-----------+-------------+--------------------+---+---------+
|customer_id|customer_name|               email|age|     city|
+-----------+-------------+--------------------+---+---------+
|        104|  Priya Patel|priya.patel@email...| 27|Ahmedabad|
|        101|  Arun Sharma|arun.sharma@email...| 28|    Delhi|
|        105|  Sneha Reddy|sneha.reddy@email...| 29|Hyderabad|
|        103|  Rahul Yadav|rahul.yadav@email...| 30|Bangalore|
|        106|   Vikas Jain|vikas.jain@email.com| 31|  Chennai|
|        102|  Meena Verma|meena.verma@email...| 34|   Mumbai|
|        107|     Amit Roy|  amit.roy@email.com| 35|  Kolkata|
+-----------+-------------+--------------------+---+---------+



## Union Operations
27. Add a new dataset for customers and perform a union operation with the customer
DataFrame.
28. Combine the sales DataFrame with another DataFrame containing additional sales
records.

In [32]:
new_customers = [
    (108, "Neha Singh", "neha.singh@email.com", 26, "Pune"),
    (109, "Rohit Gupta", "rohit.gupta@email.com", 33, "Jaipur")
]

new_customers_df = spark.createDataFrame(
    new_customers,
    ["customer_id", "customer_name", "email", "age", "city"]
)

all_cust_df = customer_df.union(new_customers_df)

all_cust_df.show()

+-----------+-------------+--------------------+---+---------+
|customer_id|customer_name|               email|age|     city|
+-----------+-------------+--------------------+---+---------+
|        101|  Arun Sharma|arun.sharma@email...| 28|    Delhi|
|        102|  Meena Verma|meena.verma@email...| 34|   Mumbai|
|        103|  Rahul Yadav|rahul.yadav@email...| 30|Bangalore|
|        104|  Priya Patel|priya.patel@email...| 27|Ahmedabad|
|        105|  Sneha Reddy|sneha.reddy@email...| 29|Hyderabad|
|        106|   Vikas Jain|vikas.jain@email.com| 31|  Chennai|
|        107|     Amit Roy|  amit.roy@email.com| 35|  Kolkata|
|        108|   Neha Singh|neha.singh@email.com| 26|     Pune|
|        109|  Rohit Gupta|rohit.gupta@email...| 33|   Jaipur|
+-----------+-------------+--------------------+---+---------+



In [35]:
sales_df.show()

new_sales = [
    (11, 106, "Mobile", 18000, "2023-10-05", "South"),
    (12, 107, "Laptop", 65000, "2023-10-12", "West")
]

new_sales_df = spark.createDataFrame(
    new_sales,
    ["sales_id", "customer_id", "product", "amount", "sale_date", "region"]
)

combined_sales_df = sales_df.union(new_sales_df)

combined_sales_df.show()

+--------+-----------+-------+------+----------+------+
|sales_id|customer_id|product|amount| sale_date|region|
+--------+-----------+-------+------+----------+------+
|       1|        101| Laptop| 50000|2023-01-15| North|
|       2|        102| Mobile| 15000|2023-02-10| South|
|       3|        103| Tablet| 20000|2023-03-05|  West|
|       4|        104| Laptop| 55000|2023-03-15|  East|
|       5|        105|Desktop| 40000|2023-04-20| North|
|       6|        101| Mobile| 15000|2023-05-10| South|
|       7|        102| Laptop| 60000|2023-06-15|  East|
|       8|        103| Tablet| 20000|2023-07-05| North|
|       9|        104|Desktop| 45000|2023-08-10|  West|
|      10|        105| Laptop| 70000|2023-09-25| North|
+--------+-----------+-------+------+----------+------+

+--------+-----------+-------+------+----------+------+
|sales_id|customer_id|product|amount| sale_date|region|
+--------+-----------+-------+------+----------+------+
|       1|        101| Laptop| 50000|2023-01-15

## Window Functions
29. Rank the sales records based on the amount column.
30. Add a cumulative sum of amount for each product in the sales DataFrame.
31. Add a column that calculates the difference between each customer's amount and the
average amount within their product group.

In [37]:
spec = Window.orderBy(col("amount").desc())

r1 = sales_df.withColumn("rank",rank().over(spec))

r1.show()

+--------+-----------+-------+------+----------+------+----+
|sales_id|customer_id|product|amount| sale_date|region|rank|
+--------+-----------+-------+------+----------+------+----+
|      10|        105| Laptop| 70000|2023-09-25| North|   1|
|       7|        102| Laptop| 60000|2023-06-15|  East|   2|
|       4|        104| Laptop| 55000|2023-03-15|  East|   3|
|       1|        101| Laptop| 50000|2023-01-15| North|   4|
|       9|        104|Desktop| 45000|2023-08-10|  West|   5|
|       5|        105|Desktop| 40000|2023-04-20| North|   6|
|       3|        103| Tablet| 20000|2023-03-05|  West|   7|
|       8|        103| Tablet| 20000|2023-07-05| North|   7|
|       2|        102| Mobile| 15000|2023-02-10| South|   9|
|       6|        101| Mobile| 15000|2023-05-10| South|   9|
+--------+-----------+-------+------+----------+------+----+



In [39]:
spec2 = Window.partitionBy("product") \
                   .orderBy("sale_date") \
                   .rowsBetween(Window.unboundedPreceding, Window.currentRow)

cum_df = sales_df.withColumn(
    "cum",
    sum(col("amount")).over(spec)
)

cum_df.show()

+--------+-----------+-------+------+----------+------+------+
|sales_id|customer_id|product|amount| sale_date|region|   cum|
+--------+-----------+-------+------+----------+------+------+
|      10|        105| Laptop| 70000|2023-09-25| North| 70000|
|       7|        102| Laptop| 60000|2023-06-15|  East|130000|
|       4|        104| Laptop| 55000|2023-03-15|  East|185000|
|       1|        101| Laptop| 50000|2023-01-15| North|235000|
|       9|        104|Desktop| 45000|2023-08-10|  West|280000|
|       5|        105|Desktop| 40000|2023-04-20| North|320000|
|       3|        103| Tablet| 20000|2023-03-05|  West|360000|
|       8|        103| Tablet| 20000|2023-07-05| North|360000|
|       2|        102| Mobile| 15000|2023-02-10| South|390000|
|       6|        101| Mobile| 15000|2023-05-10| South|390000|
+--------+-----------+-------+------+----------+------+------+



In [40]:
spec3 = Window.partitionBy("product")

avg_df = sales_df.withColumn(
    "avg", avg(col("amount")).over(spec3)
).withColumn(
    "diff", col("amount") - col("avg")
)

avg_df.show()

+--------+-----------+-------+------+----------+------+-------+-------+
|sales_id|customer_id|product|amount| sale_date|region|    avg|   diff|
+--------+-----------+-------+------+----------+------+-------+-------+
|       5|        105|Desktop| 40000|2023-04-20| North|42500.0|-2500.0|
|       9|        104|Desktop| 45000|2023-08-10|  West|42500.0| 2500.0|
|       1|        101| Laptop| 50000|2023-01-15| North|58750.0|-8750.0|
|       4|        104| Laptop| 55000|2023-03-15|  East|58750.0|-3750.0|
|       7|        102| Laptop| 60000|2023-06-15|  East|58750.0| 1250.0|
|      10|        105| Laptop| 70000|2023-09-25| North|58750.0|11250.0|
|       2|        102| Mobile| 15000|2023-02-10| South|15000.0|    0.0|
|       6|        101| Mobile| 15000|2023-05-10| South|15000.0|    0.0|
|       3|        103| Tablet| 20000|2023-03-05|  West|20000.0|    0.0|
|       8|        103| Tablet| 20000|2023-07-05| North|20000.0|    0.0|
+--------+-----------+-------+------+----------+------+-------+-

## Partitioning
32. Write the sales DataFrame to a partitioned Parquet file by region.
33. Partition the customer DataFrame by customer_city and save it as a CSV file.

In [41]:
sales_df.write \
    .partitionBy("region") \
    .mode("overwrite") \
    .parquet("/content/sales_parquet")

In [42]:
customer_df.write \
    .partitionBy("city") \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/content/customers_partitioned_csv")

## Real-World Scenarios
34. Calculate the percentage contribution of each product to the total sales.
35. Extract the year from sale_date and group by year to calculate total sales.
36. Identify the most purchased product in each region.
37. Add a column to show the difference between the highest and lowest sales for each
product.
38. Write the result of the join between sales and customer to parquet file.
39. Identify products that were sold in the last 6 months.
40. Calculate the average sales amount per customer.

In [43]:
sb_year = sales_df.withColumn('year', year('sale_date')).groupBy('year').agg(sum('amount').alias('total_sales'))

sb_year.show()

+----+-----------+
|year|total_sales|
+----+-----------+
|2023|     390000|
+----+-----------+



In [45]:
spec4 = Window.partitionBy('region').orderBy(col('total_sales').desc())

most_purchased = sales_df.groupBy('region', 'product') \
    .agg(sum('amount').alias('total_sales')) \
    .withColumn('rank', row_number().over(spec4)) \
    .filter(col('rank') == 1) \
    .drop('rank')

most_purchased.show()

+------+-------+-----------+
|region|product|total_sales|
+------+-------+-----------+
|  East| Laptop|     115000|
| North| Laptop|     120000|
| South| Mobile|      30000|
|  West|Desktop|      45000|
+------+-------+-----------+



In [47]:
product_diff = sales_df.groupBy('product') \
    .agg(
        max('amount').alias('max_sale'),
        min('amount').alias('min_sale')
    ) \
    .withColumn('difference', col('max_sale') - col('min_sale'))

product_diff.show()


+-------+--------+--------+----------+
|product|max_sale|min_sale|difference|
+-------+--------+--------+----------+
| Laptop|   70000|   50000|     20000|
| Mobile|   15000|   15000|         0|
| Tablet|   20000|   20000|         0|
|Desktop|   45000|   40000|      5000|
+-------+--------+--------+----------+



In [49]:
sales_cust = sales_df.join(customer_df, on='customer_id', how='inner')
sales_cust.write.mode('overwrite').parquet('sales_cust.parquet')

In [52]:
six_months_ago = date_sub(current_date(), 180)
recent_products = sales_df.filter(col('sale_date') >= six_months_ago) \
    .select('sales_id').distinct()

recent_products.show()

+--------+
|sales_id|
+--------+
+--------+



In [53]:
avg_sales_per_customer = sales_df.groupBy('customer_id') \
    .agg(round(sum('amount') / count('amount'), 2).alias('avg_sale'))

avg_sales_per_customer.show()

+-----------+--------+
|customer_id|avg_sale|
+-----------+--------+
|        101| 32500.0|
|        103| 20000.0|
|        102| 37500.0|
|        105| 55000.0|
|        104| 50000.0|
+-----------+--------+

